In [13]:
# RITIK + TIYA UPDATED


import json
import os
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==========================
# Load model
# ==========================
MODEL_PATH = r"D:\USAII\guardlink_final_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

# Use model labels if available
if hasattr(model.config, "id2label") and model.config.id2label:
    id2label = {int(k): v for k, v in model.config.id2label.items()}
    categories = [id2label[i] for i in range(len(id2label))]
else:
    categories = [
        "toxicity", "bullying", "threat", "sexual",
        "grooming", "sextortion", "coercion", "hate"
    ]

# ==========================
# Model inference
# ==========================
def analyze_text(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True
    )

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()

    return {categories[i]: float(probs[i]) for i in range(len(categories))}

# ==========================
# Risk computation (FIXED)
# ==========================
def compute_overall_risk(scores):
    severe_keys = ["threat", "grooming", "sextortion", "coercion"]

    severe_vals = [float(scores.get(k, 0.0)) for k in severe_keys]

    # ONLY numeric values (prevents dtype crash)
    numeric_vals = [
        float(v)
        for v in scores.values()
        if isinstance(v, (int, float, np.number))
    ]

    max_severe = max(severe_vals) if severe_vals else 0.0
    mean_all = np.mean(numeric_vals) if numeric_vals else 0.0

    risk_score = (0.7 * max_severe) + (0.3 * mean_all)
    return round(float(risk_score), 2)

# ==========================
# Sender processing
# ==========================
def process_single_sender(sender_data):
    sender_id = sender_data.get("sender_id")
    targets = sender_data.get("targets", [])

    all_scores = []  # list of per-message dicts

    for target in targets:
        for msg in target.get("conversation", []):
            text = msg.get("text", "")
            if text:
                all_scores.append(analyze_text(text))

    # Clean numeric-only output structure
    final_scores = {cat: 0.0 for cat in categories}

    if all_scores:
        for cat in categories:
            vals = [s[cat] for s in all_scores if cat in s]
            final_scores[cat] = round(float(np.mean(vals)), 4) if vals else 0.0

    risk_score = compute_overall_risk(final_scores)

    return {
        "bert_analysis": {
            "sender_id": sender_id,
            **final_scores,
            "risk-score": risk_score
        }
    }

# ==========================
# Main execution
# ==========================
input_file_path = "comments_raw.jsonl"
output_file_path = "sender_analysis_Report.jsonl"

if os.path.exists(input_file_path):
    print(f"Processing {input_file_path}...")

    with open(input_file_path, "r", encoding="utf-8") as infile, \
         open(output_file_path, "w", encoding="utf-8") as outfile:

        for i, line in enumerate(infile):
            line = line.strip()
            if not line:
                continue

            sender_data = json.loads(line)
            result = process_single_sender(sender_data)

            outfile.write(json.dumps(result) + "\n")

            if i % 10 == 0:
                print(f"Processed {i} senders")

    print(f"Done → {output_file_path}")

else:
    print("Input file not found")

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Processing comments_raw.jsonl...
Processed 0 senders
Processed 10 senders
Processed 20 senders
Processed 30 senders
Processed 40 senders
Processed 50 senders
Processed 60 senders
Processed 70 senders
Processed 80 senders
Processed 90 senders
Processed 100 senders
Processed 110 senders
Processed 120 senders
Processed 130 senders
Processed 140 senders
Processed 150 senders
Processed 160 senders
Processed 170 senders
Processed 180 senders
Processed 190 senders
Processed 200 senders
Processed 210 senders
Processed 220 senders
Processed 230 senders
Processed 240 senders
Processed 250 senders
Processed 260 senders
Processed 270 senders
Processed 280 senders
Processed 290 senders
Processed 300 senders
Processed 310 senders
Processed 320 senders
Processed 330 senders
Processed 340 senders
Processed 350 senders
Processed 360 senders
Processed 370 senders
Processed 380 senders
Processed 390 senders
Processed 400 senders
Processed 410 senders
Processed 420 senders
Processed 430 senders
Processed 

In [14]:
import json
import os
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==========================
# Load model
# ==========================
MODEL_PATH = r"D:\USAII\guardlink_final_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

# Labels
if hasattr(model.config, "id2label") and model.config.id2label:
    id2label = {int(k): v for k, v in model.config.id2label.items()}
    categories = [id2label[i] for i in range(len(id2label))]
else:
    categories = [
        "toxicity", "bullying", "threat", "sexual",
        "grooming", "sextortion", "coercion", "hate"
    ]

# ==========================
# Inference
# ==========================
def analyze_text(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True
    )

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()

    return {categories[i]: float(probs[i]) for i in range(len(categories))}

# ==========================
# Risk computation (FIXED + SCALED)
# ==========================
def compute_risk(threat, bullying, sextortion, hate,
                 toxicity, grooming, sexual, coercion):

    severe = (
        0.30 * threat +
        0.25 * bullying +
        0.25 * sextortion +
        0.20 * hate
    )

    general = np.mean([toxicity, grooming, sexual, coercion])

    raw = 0.8 * severe + 0.2 * general

    risk = 0.3 + 0.65 * np.sqrt(raw)

    return round(risk, 4)


# ==========================
# Sender processing
# ==========================
def process_single_sender(sender_data):
    sender_id = sender_data.get("sender_id")
    targets = sender_data.get("targets", [])

    all_scores = []

    for target in targets:
        for msg in target.get("conversation", []):
            text = msg.get("text", "")
            if text:
                all_scores.append(analyze_text(text))

    final_scores = {cat: 0.0 for cat in categories}

    if all_scores:
        for cat in categories:
            final_scores[cat] = float(np.mean([s[cat] for s in all_scores]))

    risk_score = compute_overall_risk(final_scores)

    return {
        "bert_analysis": {
            "sender_id": sender_id,
            **{k: round(v, 4) for k, v in final_scores.items()},
            "risk-score": risk_score
        }
    }

# ==========================
# Main execution
# ==========================
input_file_path = "comments_raw.jsonl"
output_file_path = "sender_analysis_Report.jsonl"

if os.path.exists(input_file_path):
    print(f"Processing {input_file_path}...")

    with open(input_file_path, "r", encoding="utf-8") as infile, \
         open(output_file_path, "w", encoding="utf-8") as outfile:

        for i, line in enumerate(infile):
            line = line.strip()
            if not line:
                continue

            sender_data = json.loads(line)
            result = process_single_sender(sender_data)

            outfile.write(json.dumps(result) + "\n")

            if i % 10 == 0:
                print(f"Processed {i} senders")

    print(f"Done → {output_file_path}")

else:
    print("Input file not found")

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Processing comments_raw.jsonl...
Processed 0 senders
Processed 10 senders
Processed 20 senders
Processed 30 senders
Processed 40 senders
Processed 50 senders
Processed 60 senders
Processed 70 senders
Processed 80 senders
Processed 90 senders
Processed 100 senders
Processed 110 senders
Processed 120 senders
Processed 130 senders
Processed 140 senders
Processed 150 senders
Processed 160 senders
Processed 170 senders
Processed 180 senders
Processed 190 senders
Processed 200 senders
Processed 210 senders
Processed 220 senders
Processed 230 senders
Processed 240 senders
Processed 250 senders
Processed 260 senders
Processed 270 senders
Processed 280 senders
Processed 290 senders
Processed 300 senders
Processed 310 senders
Processed 320 senders
Processed 330 senders
Processed 340 senders
Processed 350 senders
Processed 360 senders
Processed 370 senders
Processed 380 senders
Processed 390 senders
Processed 400 senders
Processed 410 senders
Processed 420 senders
Processed 430 senders
Processed 

In [15]:
import json
import torch
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =========================
# CONFIG
# =========================

MODEL_PATH = r"D:\USAII\guardlink_final_model"
INPUT_FILE = "comments_raw.jsonl"
OUTPUT_FILE = "Comments_processed.jsonl"

CATEGORIES = [
    "toxicity",
    "bullying",
    "threat",
    "sexual",
    "grooming",
    "sextortion",
    "coercion",
    "hate"
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# =========================
# LOAD MODEL
# =========================

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.to(DEVICE)
model.eval()
print("Model loaded.")

# =========================
# INFERENCE
# =========================

def run_inference(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits.squeeze()

    # sigmoid for multi-label, softmax for single-label
    # guardlink is multi-label so using sigmoid
    scores = torch.sigmoid(logits).cpu().numpy()

    # if model has fewer labels than CATEGORIES, pad with zeros
    if len(scores) < len(CATEGORIES):
        scores = np.pad(scores, (0, len(CATEGORIES) - len(scores)))

    return scores

# =========================
# PROCESS FILE
# =========================

print(f"Reading from: {INPUT_FILE}")
results = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

print(f"Total comments to process: {len(records)}")

for i, record in enumerate(records):
    text = record["text"]
    scores = run_inference(text)

    # map scores to category names
    category_scores = {
        cat: round(float(scores[j]), 4)
        for j, cat in enumerate(CATEGORIES)
    }

    # confidence = max score across all categories
    confidence = round(float(np.max(scores)), 4)

    bert_entry = {
        "bert_analysis": {
            "receiver_id": record["comment_id"],
            "sender_id": record["sender"]["user_id"],
            **category_scores,
            "confidence": confidence
        }
    }

    results.append(bert_entry)

    if (i + 1) % 50 == 0:
        print(f"  Processed {i + 1}/{len(records)}")

# =========================
# SAVE
# =========================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for entry in results:
        f.write(json.dumps(entry) + "\n")

print(f"\nDONE. Saved {len(results)} records to {OUTPUT_FILE}")

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Using device: cuda
Loading model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded.
Reading from: comments_raw.jsonl
Total comments to process: 2700
  Processed 50/2700
  Processed 100/2700
  Processed 150/2700
  Processed 200/2700
  Processed 250/2700
  Processed 300/2700
  Processed 350/2700
  Processed 400/2700
  Processed 450/2700
  Processed 500/2700
  Processed 550/2700
  Processed 600/2700
  Processed 650/2700
  Processed 700/2700
  Processed 750/2700
  Processed 800/2700
  Processed 850/2700
  Processed 900/2700
  Processed 950/2700
  Processed 1000/2700
  Processed 1050/2700
  Processed 1100/2700
  Processed 1150/2700
  Processed 1200/2700
  Processed 1250/2700
  Processed 1300/2700
  Processed 1350/2700
  Processed 1400/2700
  Processed 1450/2700
  Processed 1500/2700
  Processed 1550/2700
  Processed 1600/2700
  Processed 1650/2700
  Processed 1700/2700
  Processed 1750/2700
  Processed 1800/2700
  Processed 1850/2700
  Processed 1900/2700
  Processed 1950/2700
  Processed 2000/2700
  Processed 2050/2700
  Processed 2100/2700
  Processed 2150/